# FLAG 2027 - Notebook 32c: VoxCeleb2 part 4 (`vox2_mp4_4.zip`, ~1,100 more identities), all seven encoders

Same notebook as NB32, pointed at the next VoxCeleb2 dev part. Part 2 moved dev by -0.24 (E 22.16 -> E2 21.92) with gains in 3 of 4 cells, so the direction is right but one part is inside noise. Two more parts together may cross the 0.5 adoption line. Identical recipe: 20 clips per identity, one face frame at 1.0 s, 16 kHz audio, **Bengali removed** by VoxLingua107 language-ID, encoders identical to NB20 Cell 4, files saved as `features/vox_vox2_mp4_4.*`.

About 4-5 h on a T4; resumable per encoder. Good for an overnight run. Needs `HF_TOKEN` in Colab Secrets. Paste the **SUMMARY**. (After this, NB20 gets a system trained on 4 VoxCeleb parts.)


In [ ]:
# Cell 1 - Drive, GPU (hard check), packages, HF token
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub onnx onnx2torch
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
HF_TOKEN = userdata.get('HF_TOKEN')
print('[NB32c v1 5-Nov part 4] Device:', torch.cuda.get_device_name(0), '| HF token found:', HF_TOKEN is not None, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - download vox2_mp4_4, select 20 clips / identity (NB07 rule), ffmpeg, Bengali filter, ECAPA sanity -> work dir + row table
import os, re, json, time, zipfile, subprocess, glob
import numpy as np, pandas as pd
import librosa, soundfile as sf
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, list_repo_files
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); CACHE = os.path.join(PROJECT, 'models_cache')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
ZNAME, CPI, TAG = 'vox2/vox2_mp4_4.zip', 20, 'vox2_mp4_4'
W = '/content/vox_work/' + TAG; os.makedirs(W, exist_ok=True)
ROWS = os.path.join(FEATDIR, 'vox_%s_rows.npz' % TAG)          # spk + relative wav/jpg names after the Bengali filter (lets a restart skip this cell)
OLD_SPK = set()
for t in ['vox2_test_mp4', 'vox2_mp4_1', 'vox2_mp4_2', 'vox2_mp4_3']:
    OLD_SPK |= set(np.load(os.path.join(FEATDIR, 'vox_%s.npz' % t))['spk'])
try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max()); b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def ffmpeg_audio(job):
    mp4, wav, jpg = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', jpg]).returncode
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(jpg)
if os.path.exists(ROWS) and os.path.exists(W + '/done'):
    r = np.load(ROWS); SPK, WAVS, JPGS = r['spk'], [os.path.join(W, x) for x in r['wav']], [os.path.join(W, x) for x in r['jpg']]
    print('rows loaded:', len(SPK))
else:
    t0 = time.time()
    files = list_repo_files('ProgramComputer/voxceleb', repo_type='dataset', token=HF_TOKEN)
    assert ZNAME in files, 'zip %s not on the mirror; available vox2 zips: %s - tell Claude' % (ZNAME, [f for f in files if f.startswith('vox2/')])
    zpath = hf_hub_download('ProgramComputer/voxceleb', ZNAME, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    zf = zipfile.ZipFile(zpath); tree = {}
    for n in zf.namelist():
        mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
        if mm:
            tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
    overlap = {'vox_' + p for p in tree} & OLD_SPK
    note('%s: %d identities in the zip | %d overlap with the identities already used (dropped)' % (TAG, len(tree), len(overlap)))
    rng = np.random.default_rng(0); picked = []
    for pid, vids in tree.items():
        pools = [list(rng.permutation(v)) for v in vids.values()]; chosen = []
        while len(chosen) < CPI and any(pools):
            for p in pools:
                if p and len(chosen) < CPI:
                    chosen.append(p.pop())
        if 'vox_' + pid not in OLD_SPK:
            picked += [(pid, c) for c in chosen]
    jobs = []
    for i, (pid, member) in enumerate(picked):
        jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i), '%s/%06d.jpg' % (W, i)))
    zf.close(); os.remove(zpath)
    with ThreadPoolExecutor(8) as ex:
        ok = list(tqdm(ex.map(ffmpeg_audio, jobs), total=len(jobs), desc='ffmpeg'))
    shutil.rmtree(W + '/mp4', ignore_errors=True)
    keep = [i for i, g in enumerate(ok) if g]
    wavs = [jobs[i][1] for i in keep]; jpgs = [jobs[i][2] for i in keep]; spk = np.array(['vox_' + picked[i][0] for i in keep])
    LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})
    lang = [''] * len(wavs)
    for idx in tqdm(list(batches_by_length(wavs, 32)), desc='language-ID'):
        b, rl = pad(wavs, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            lang[j] = lab.split(':')[0].strip()
    lang = np.array(lang); nb = lang != 'bn'
    del LID; torch.cuda.empty_cache()
    SPK = spk[nb]; WAVS = [w for w, k in zip(wavs, nb) if k]; JPGS = [j for j, k in zip(jpgs, nb) if k]
    np.savez(ROWS, spk=SPK, wav=np.array([os.path.basename(w) for w in WAVS]), jpg=np.array([os.path.basename(j) for j in JPGS]), lang=lang[nb])
    open(W + '/done', 'w').close()
    top = pd.Series(lang).value_counts().head(6)
    note('%s: %d clips kept from %d identities | Bengali clips removed: %d | top languages: %s | %.0f min' % (TAG, len(SPK), len(set(SPK)), int((~nb).sum()), ', '.join('%s %d' % (k, v) for k, v in top.items()), (time.time() - t0) / 60))
assert len(set(SPK) & OLD_SPK) == 0

In [ ]:
# Cell 3 - the seven encoders (definitions identical to NB20 Cell 4), each saved as soon as it finishes (RESUMABLE per encoder)
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
def enc_ecapa(paths, bs=16):
    E = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); E.eval()
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = E.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out
def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(name); out = []
    for i in tqdm(range(0, len(paths), 64), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + 64]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(model.visual_projection(model.vision_model(pixel_values=x).pooler_output).float().cpu().numpy())
    return np.concatenate(out)
def enc_wavlm(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name); model = WavLMForXVector.from_pretrained(name).to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), 32), desc='wavlm'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + 32]]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)
def enc_r293(paths, sec=6, bs=32):
    import torchaudio
    from onnx2torch import convert
    repo = 'Wespeaker/wespeaker-voxceleb-resnet293-LM'
    onnx_name = [f for f in list_repo_files(repo) if f.endswith('.onnx')][0]
    net = convert(hf_hub_download(repo, onnx_name, local_dir=os.path.join(CACHE, 'wespeaker_r293'))).to(dev).eval(); L = 16000 * sec
    def fbank(p):
        w = librosa.load(p, sr=16000, mono=True)[0]
        w = w[(len(w) - L) // 2:(len(w) - L) // 2 + L] if len(w) >= L else np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
        fb = torchaudio.compliance.kaldi.fbank(torch.from_numpy(w).unsqueeze(0) * (1 << 15), num_mel_bins=80, frame_length=25, frame_shift=10, dither=0.0, sample_frequency=16000, window_type='hamming', use_energy=False)
        return fb - fb.mean(0, keepdim=True)
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='resnet293'):
        x = torch.stack([fbank(p) for p in paths[i:i + bs]]).to(dev)
        with torch.no_grad():
            e = net(x)
        e = e[0] if isinstance(e, (list, tuple)) else e; out.append(e.reshape(len(x), -1).float().cpu().numpy())
    return np.concatenate(out)
def enc_redim(paths, bs=16, max_sec=10):
    net = torch.hub.load('PalabraAI/redimnet2', 'redimnet2', model_name='b6', train_type='lm', dataset='vb2+vox2_v0', pretrained=True).to(dev).eval()
    order = np.argsort([sf.info(p).duration for p in paths]); emb = {}
    for i in tqdm(range(0, len(order), bs), desc='redimnet2'):
        idx = order[i:i + bs]; wavs = [librosa.load(paths[j], sr=16000, mono=True)[0][:16000 * max_sec] for j in idx]
        L = max(len(w) for w in wavs); b = torch.zeros(len(wavs), L)
        for k_, w in enumerate(wavs):
            b[k_] = torch.from_numpy(np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L])
        with torch.no_grad():
            e = net(b.to(dev))
        e = e[0] if isinstance(e, (list, tuple)) else e
        for k_, j in enumerate(idx):
            emb[int(j)] = e[k_].float().cpu().numpy().reshape(-1)
    return np.stack([emb[j] for j in range(len(paths))])
from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2PreTrainedModel
class ModelHead(nn.Module):
    def __init__(self, hidden, num_labels, drop):
        super().__init__(); self.dense = nn.Linear(hidden, hidden); self.dropout = nn.Dropout(drop); self.out_proj = nn.Linear(hidden, num_labels)
    def forward(self, x):
        return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))
class AgeGenderModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config); self.wav2vec2 = Wav2Vec2Model(config)
        self.age = ModelHead(config.hidden_size, 1, config.final_dropout); self.gender = ModelHead(config.hidden_size, 3, config.final_dropout); self.post_init()
    def forward(self, input_values):
        h = self.wav2vec2(input_values)[0].mean(1); return h, self.age(h), torch.softmax(self.gender(h), dim=1)
def enc_audeer(paths, bs=16, max_sec=8):
    name = 'audeering/wav2vec2-large-robust-24-ft-age-gender'; proc = Wav2Vec2Processor.from_pretrained(name)
    try:
        model = AgeGenderModel.from_pretrained(name)
    except Exception:
        base = Wav2Vec2Model.from_pretrained(name); model = AgeGenderModel(base.config); model.wav2vec2 = base
        try:
            from safetensors.torch import load_file; sd = load_file(hf_hub_download(name, 'model.safetensors'))
        except Exception:
            sd = torch.load(hf_hub_download(name, 'pytorch_model.bin'), map_location='cpu')
        model.load_state_dict({k: v for k, v in sd.items() if k.startswith(('age.', 'gender.'))}, strict=False)
    model = model.to(dev).eval().half(); H, AG = [], []
    for i in tqdm(range(0, len(paths), bs), desc='audeering'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + bs]]
        x = proc(wavs, sampling_rate=16000, padding=True, return_tensors='pt')['input_values'].to(dev).half()
        with torch.no_grad():
            h, a, g = model(x)
        H.append(h.float().cpu().numpy()); AG.append(torch.cat([a, g], 1).float().cpu().numpy())
    return np.concatenate(H), np.concatenate(AG)
def enc_whisper(paths, bs=8, layers=(20, 29)):
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    fe = WhisperFeatureExtractor.from_pretrained(name); enc = WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), bs), desc='whisper-v3 enc'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths[i:i + bs]]
        x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
        with torch.no_grad():
            hs = enc(x, output_hidden_states=True).hidden_states
        h = torch.stack(hs[layers[0]:layers[1]]).float().mean(0)
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); out.append(h[k_, :n].mean(0).cpu().numpy())
    del enc; torch.cuda.empty_cache(); return np.stack(out)
PART = lambda n: os.path.join(FEATDIR, 'vox_%s_part_%s.npy' % (TAG, n))       # per-encoder checkpoint
JOBS = [('clip', lambda: enc_clip(JPGS)), ('ecapa', lambda: enc_ecapa(WAVS)), ('wavlm_sv', lambda: enc_wavlm(WAVS)), ('r293', lambda: enc_r293(WAVS)),
        ('audeer', None), ('redim', lambda: enc_redim(WAVS)), ('whisper', lambda: enc_whisper(WAVS))]
for name, fn in JOBS:
    if os.path.exists(PART(name)):
        print(name, 'already done'); continue
    t0 = time.time()
    if name == 'audeer':
        H, AG = enc_audeer(WAVS); np.save(PART('audeer_ag'), AG.astype(np.float32)); E = H
    else:
        E = fn()
    E = np.asarray(E, np.float32); assert len(E) == len(SPK); np.save(PART(name), E); torch.cuda.empty_cache()
    note('%-8s %d x %d in %.0f min' % (name, E.shape[0], E.shape[1], (time.time() - t0) / 60))

In [ ]:
# Cell 4 - write the final files in the SAME layout as the existing VoxCeleb parts, sanity-check identity signal, SUMMARY
P = {n: np.load(PART(n)) for n in ['clip', 'ecapa', 'wavlm_sv', 'r293', 'audeer', 'redim', 'whisper']}
np.savez(os.path.join(FEATDIR, 'vox_%s.npz' % TAG), spk=SPK, clip=P['clip'], ecapa=P['ecapa'], wavlm_sv=P['wavlm_sv'])
np.save(os.path.join(FEATDIR, 'vox_%s_r293.npy' % TAG), P['r293']); np.save(os.path.join(FEATDIR, 'vox_%s_redim.npy' % TAG), P['redim'])
np.savez(os.path.join(FEATDIR, 'vox_%s_audeer.npz' % TAG), h=P['audeer'], ag=np.load(PART('audeer_ag')))
np.save(os.path.join(FEATDIR, 'vox_%s_whisper.npy' % TAG), P['whisper'])
# identity sanity on each block: same-identity cosine must exceed different-identity cosine (compare with the old parts)
def gap(E, S, n=600, seed=0):
    r = np.random.default_rng(seed); idx = r.choice(len(S), min(n, len(S)), replace=False); Z = E[idx] / (np.linalg.norm(E[idx], axis=1, keepdims=True) + 1e-9); C = Z @ Z.T
    same = (S[idx][:, None] == S[idx][None, :]) & ~np.eye(len(idx), dtype=bool); diff = S[idx][:, None] != S[idx][None, :]
    return C[same].mean() - C[diff].mean()
old = dict(np.load(os.path.join(FEATDIR, 'vox_vox2_mp4_1.npz')))
old['whisper'] = np.load(os.path.join(FEATDIR, 'vox_vox2_mp4_1_whisper.npy')); old['r293'] = np.load(os.path.join(FEATDIR, 'vox_vox2_mp4_1_r293.npy'))
note('identity gap (same - different cosine), new part vs vox2_mp4_1: ' + ', '.join('%s %.3f/%.3f' % (n, gap(P[n], SPK), gap(old[n], old['spk'])) for n in ['clip', 'ecapa', 'wavlm_sv', 'r293', 'whisper']))
note('[NB32c v1 5-Nov] %s ready: %d rows, %d identities | files: vox_%s.npz (+ _r293, _redim, _audeer, _whisper)' % (TAG, len(SPK), len(set(SPK)), TAG))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)